# Training on image and audio

Same architecture, trained on CIFAR-10 and `waves_yesno` instead of text. Both datasets are a few MB, well below enwik8's scale - small-scale runs completing the three-modality comparison in the paper's Table 1.

**GPU**: `Runtime > Change runtime type > T4 GPU`

Upload `colab_data_bundle_v3.zip` (`cifar_images` and `multimodal_data/audio`).

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import time
import os

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
if device == 'cpu':
    print('Warning: GPU is not enabled!')
torch.manual_seed(11)

device: cuda


In [2]:
from google.colab import files
uploaded = files.upload()  # colab_data_bundle_v3.zip
!unzip -o colab_data_bundle_v3.zip -d /content/data > /dev/null
print('ready')

Saving colab_data_bundle_v3.zip to colab_data_bundle_v3.zip
ready


## Architecture
Same small GPT as before, kept small given the limited data.

In [3]:
VOCAB_SIZE = 256
BLOCK_SIZE = 256
N_LAYER = 4
N_HEAD = 4
N_EMBD = 128
DROPOUT = 0.1

class CausalSelfAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.qkv = nn.Linear(N_EMBD, 3*N_EMBD)
        self.proj = nn.Linear(N_EMBD, N_EMBD)
        self.n_head = N_HEAD
        self.drop = nn.Dropout(DROPOUT)
        mask = torch.tril(torch.ones(BLOCK_SIZE, BLOCK_SIZE)).view(1, 1, BLOCK_SIZE, BLOCK_SIZE)
        self.register_buffer('mask', mask)

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.split(N_EMBD, dim=2)
        hd = C // self.n_head
        q = q.view(B, T, self.n_head, hd).transpose(1, 2)
        k = k.view(B, T, self.n_head, hd).transpose(1, 2)
        v = v.view(B, T, self.n_head, hd).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) / (hd ** 0.5)
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        att = self.drop(att)
        out = att @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(out)

class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln1 = nn.LayerNorm(N_EMBD)
        self.attn = CausalSelfAttention()
        self.ln2 = nn.LayerNorm(N_EMBD)
        self.mlp = nn.Sequential(
            nn.Linear(N_EMBD, 4*N_EMBD), nn.GELU(),
            nn.Linear(4*N_EMBD, N_EMBD), nn.Dropout(DROPOUT)
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class TinyGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok_emb = nn.Embedding(VOCAB_SIZE, N_EMBD)
        self.pos_emb = nn.Parameter(torch.zeros(1, BLOCK_SIZE, N_EMBD))
        self.drop = nn.Dropout(DROPOUT)
        self.blocks = nn.Sequential(*[Block() for _ in range(N_LAYER)])
        self.ln_f = nn.LayerNorm(N_EMBD)
        self.head = nn.Linear(N_EMBD, VOCAB_SIZE, bias=False)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.tok_emb(idx) + self.pos_emb[:, :T, :]
        x = self.drop(x)
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, VOCAB_SIZE), targets.view(-1), reduction='mean')
        return logits, loss

def get_batch(data_split, batch_size):
    ix = torch.randint(0, len(data_split) - BLOCK_SIZE - 1, (batch_size,))
    x = torch.stack([data_split[i:i+BLOCK_SIZE] for i in ix])
    y = torch.stack([data_split[i+1:i+BLOCK_SIZE+1] for i in ix])
    return x.to(device), y.to(device)

def train_model(train_data, steps=6000, lr=3e-4, batch_size=64):
    model = TinyGPT().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    t0 = time.time()
    model.train()
    for step in range(steps):
        xb, yb = get_batch(train_data, batch_size)
        _, loss = model(xb, yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        if step % 200 == 0 or step == steps - 1:
            print(f'step {step:5d}  loss {loss.item():.4f}  bits/byte {loss.item()/0.6931:.4f}  '
                  f'({time.time()-t0:.0f}s)')
    print('training done:', round(time.time()-t0), 's')
    return model

def extract_losses(model, heldout_data):
    model.eval()
    n_chunks = len(heldout_data) // BLOCK_SIZE
    per_position_bits = np.zeros(n_chunks * (BLOCK_SIZE - 1), dtype=np.float32)
    with torch.no_grad():
        for c in range(n_chunks):
            chunk = heldout_data[c*BLOCK_SIZE:(c+1)*BLOCK_SIZE]
            x = chunk[:-1].unsqueeze(0).to(device)
            y = chunk[1:].unsqueeze(0).to(device)
            logits, _ = model(x)
            logp = F.log_softmax(logits, dim=-1)
            token_logp = logp.gather(-1, y.unsqueeze(-1)).squeeze(-1).squeeze(0)
            bits = (-token_logp / np.log(2)).cpu().numpy()
            start = c*(BLOCK_SIZE-1)
            per_position_bits[start:start+(BLOCK_SIZE-1)] = bits
    print('mean bits/byte on held-out:', per_position_bits.mean())
    return per_position_bits, n_chunks

## Image (CIFAR-10)
20% of images held out entirely; the rest flattened and concatenated for training.

In [4]:
from PIL import Image

IMG_DIR = '/content/data/cifar_images'
files_img = sorted(os.listdir(IMG_DIR))
n_heldout_imgs = max(1, len(files_img)//5)
heldout_files = files_img[:n_heldout_imgs]
train_files = files_img[n_heldout_imgs:]
print('train images:', len(train_files), ' heldout images:', len(heldout_files))

def load_and_flatten(file_list):
    arrs = []
    for f in file_list:
        im = np.array(Image.open(os.path.join(IMG_DIR, f)))
        arrs.append(im.flatten())
    return np.concatenate(arrs)

img_train_bytes = load_and_flatten(train_files)
img_heldout_bytes = load_and_flatten(heldout_files)
print('train bytes:', len(img_train_bytes), ' heldout bytes:', len(img_heldout_bytes))

img_train_data = torch.tensor(img_train_bytes, dtype=torch.long)
img_heldout_data = torch.tensor(img_heldout_bytes, dtype=torch.long)

train images: 48  heldout images: 12
train bytes: 786432  heldout bytes: 196608


In [5]:
img_model = train_model(img_train_data, steps=6000)

step     0  loss 5.7084  bits/byte 8.2360  (1s)
step   200  loss 2.9277  bits/byte 4.2240  (15s)
step   400  loss 2.7464  bits/byte 3.9625  (30s)
step   600  loss 2.6121  bits/byte 3.7688  (45s)
step   800  loss 2.4549  bits/byte 3.5420  (60s)
step  1000  loss 2.4476  bits/byte 3.5314  (75s)
step  1200  loss 2.3338  bits/byte 3.3671  (90s)
step  1400  loss 2.2865  bits/byte 3.2989  (106s)
step  1600  loss 2.1174  bits/byte 3.0550  (122s)
step  1800  loss 2.1913  bits/byte 3.1615  (137s)
step  2000  loss 2.1052  bits/byte 3.0374  (153s)
step  2200  loss 2.0709  bits/byte 2.9879  (169s)
step  2400  loss 2.0183  bits/byte 2.9120  (185s)
step  2600  loss 2.0687  bits/byte 2.9847  (201s)
step  2800  loss 2.0069  bits/byte 2.8955  (217s)
step  3000  loss 2.0281  bits/byte 2.9261  (234s)
step  3200  loss 1.9630  bits/byte 2.8322  (251s)
step  3400  loss 2.0030  bits/byte 2.8900  (268s)
step  3600  loss 1.9814  bits/byte 2.8588  (285s)
step  3800  loss 1.9462  bits/byte 2.8079  (303s)
step  40

In [6]:
img_losses, img_n_chunks = extract_losses(img_model, img_heldout_data)
np.save('neural_lm_image_losses.npy', img_losses)
np.save('neural_lm_image_meta.npy', np.array([BLOCK_SIZE, img_n_chunks, N_LAYER, N_HEAD, N_EMBD,
                                                sum(p.numel() for p in img_model.parameters())]))
from google.colab import files
files.download('neural_lm_image_losses.npy')
files.download('neural_lm_image_meta.npy')

mean bits/byte on held-out: 2.829425


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Shuffled control (image model)

Run right here while `img_model` and `img_heldout_data` are still in memory.

In [7]:
def spectrum_zscore(cost_signal, target_period):
    sig = cost_signal - cost_signal.mean()
    spec = np.abs(np.fft.rfft(sig))
    freqs = np.fft.rfftfreq(len(sig), d=1.0)
    idx = np.argmin(np.abs(freqs - 1.0/target_period))
    others = np.delete(spec[1:], idx-1)
    z = (spec[idx]-others.mean())/others.std()
    return z

# Tested at the row-width period (128), the real claimed structure -- NOT the context-reset
# period (BLOCK_SIZE-1), which is an evaluation-procedure artifact and is not expected to
# change under shuffling (see the paper, Section 6.2.1).
ROW_WIDTH = 128
z_img_real = spectrum_zscore(img_losses, ROW_WIDTH)

img_heldout_shuf = img_heldout_data[torch.randperm(len(img_heldout_data))]
img_shuf_losses, _ = extract_losses(img_model, img_heldout_shuf)
z_img_shuf = spectrum_zscore(img_shuf_losses, ROW_WIDTH)

print(f'real held-out images:    mean={img_losses.mean():.4f} bits/byte, row-width z={z_img_real:.2f}')
print(f'shuffled control:        mean={img_shuf_losses.mean():.4f} bits/byte, row-width z={z_img_shuf:.2f}')

mean bits/byte on held-out: 14.031882
real held-out images:    mean=2.8294 bits/byte, row-width z=15.97
shuffled control:        mean=14.0319 bits/byte, row-width z=0.48


### Optional: full-scale audio

At the small scale (8 recordings, 1 held out), the audio held-out set is too short for a meaningful shuffled control. To run the audio controls below at the paper's full scale, upload `audio_bundle_full60.zip` here first; it overwrites the smaller audio folder from the bundle above (60 recordings, 12 held out). Skip this cell to keep the small-scale run.

In [8]:
from google.colab import files
uploaded = files.upload()  # audio_bundle_full60.zip
!unzip -o audio_bundle_full60.zip -d /content/data > /dev/null
print('full-scale audio ready (60 recordings, 12 held out)')

Saving audio_bundle_full60.zip to audio_bundle_full60.zip
full-scale audio ready (60 recordings, 12 held out)


## Audio (waves_yesno)
Quantized to 256 levels; 20% of files held out entirely.

In [9]:
import wave

AUDIO_DIR = '/content/data/multimodal_data/audio'
files_audio = sorted(os.listdir(AUDIO_DIR))
n_heldout_audio = max(1, len(files_audio)//5)
heldout_audio_files = files_audio[:n_heldout_audio]
train_audio_files = files_audio[n_heldout_audio:]
print('train audio files:', len(train_audio_files), ' heldout:', len(heldout_audio_files))

def load_and_quantize(file_list):
    arrs = []
    for f in file_list:
        w = wave.open(os.path.join(AUDIO_DIR, f), 'rb')
        d = np.frombuffer(w.readframes(w.getnframes()), dtype=np.int16)
        q = ((d.astype(int) + 32768) * 256 // 65536).clip(0, 255)
        arrs.append(q)
    return np.concatenate(arrs)

audio_train_bytes = load_and_quantize(train_audio_files)
audio_heldout_bytes = load_and_quantize(heldout_audio_files)
print('train samples:', len(audio_train_bytes), ' heldout samples:', len(audio_heldout_bytes))

audio_train_data = torch.tensor(audio_train_bytes, dtype=torch.long)
audio_heldout_data = torch.tensor(audio_heldout_bytes, dtype=torch.long)

train audio files: 48  heldout: 12
train samples: 2337920  heldout samples: 603440


In [10]:
audio_model = train_model(audio_train_data, steps=6000)

step     0  loss 5.8166  bits/byte 8.3922  (0s)
step   200  loss 1.1083  bits/byte 1.5991  (16s)
step   400  loss 1.1490  bits/byte 1.6577  (32s)
step   600  loss 1.1765  bits/byte 1.6974  (48s)
step   800  loss 1.4420  bits/byte 2.0806  (65s)
step  1000  loss 1.2541  bits/byte 1.8094  (81s)
step  1200  loss 1.0509  bits/byte 1.5162  (98s)
step  1400  loss 1.0507  bits/byte 1.5159  (116s)
step  1600  loss 0.9469  bits/byte 1.3661  (133s)
step  1800  loss 0.9410  bits/byte 1.3577  (151s)
step  2000  loss 0.9843  bits/byte 1.4202  (168s)
step  2200  loss 0.9982  bits/byte 1.4402  (186s)
step  2400  loss 1.2031  bits/byte 1.7359  (204s)
step  2600  loss 1.0613  bits/byte 1.5312  (222s)
step  2800  loss 0.9682  bits/byte 1.3968  (240s)
step  3000  loss 1.0288  bits/byte 1.4843  (258s)
step  3200  loss 1.1158  bits/byte 1.6098  (276s)
step  3400  loss 1.0887  bits/byte 1.5708  (293s)
step  3600  loss 0.8889  bits/byte 1.2824  (311s)
step  3800  loss 0.9798  bits/byte 1.4136  (329s)
step  40

In [11]:
audio_losses, audio_n_chunks = extract_losses(audio_model, audio_heldout_data)
np.save('neural_lm_audio_losses.npy', audio_losses)
np.save('neural_lm_audio_meta.npy', np.array([BLOCK_SIZE, audio_n_chunks, N_LAYER, N_HEAD, N_EMBD,
                                                sum(p.numel() for p in audio_model.parameters())]))
from google.colab import files
files.download('neural_lm_audio_losses.npy')
files.download('neural_lm_audio_meta.npy')
print('four files downloaded (2 for image, 2 for audio) - please send me all of them')

mean bits/byte on held-out: 1.3259006


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

four files downloaded (2 for image, 2 for audio) - please send me all of them


### Shuffled control (audio model)

Run right here while `audio_model` and `audio_heldout_data` are still in memory.

In [12]:
# Tested at the pitch band, the real claimed structure -- NOT the context-reset period
# (BLOCK_SIZE-1), which is an evaluation-procedure artifact and is not expected to change
# under shuffling. The pitch period isn't a fixed single frequency, so we search a band
# (40-200 samples, the typical human pitch range) rather than testing one fixed target.
def pitch_band_zscore(cost_signal):
    sig = cost_signal - cost_signal.mean()
    spec = np.abs(np.fft.rfft(sig))
    freqs = np.fft.rfftfreq(len(sig), d=1.0)
    band = (freqs > 1/200) & (freqs < 1/40)
    top_idx = np.argmax(spec[band])
    z = (spec[band][top_idx] - spec[1:].mean()) / spec[1:].std()
    return z

z_audio_real = pitch_band_zscore(audio_losses)

audio_heldout_shuf = audio_heldout_data[torch.randperm(len(audio_heldout_data))]
audio_shuf_losses, _ = extract_losses(audio_model, audio_heldout_shuf)
z_audio_shuf = pitch_band_zscore(audio_shuf_losses)

print(f'real held-out audio:     mean={audio_losses.mean():.4f} bits/byte, pitch-band z={z_audio_real:.2f}')
print(f'shuffled control:        mean={audio_shuf_losses.mean():.4f} bits/byte, pitch-band z={z_audio_shuf:.2f}')
print()
print('If you uploaded audio_bundle_full60.zip above, this is the full-scale result the paper needs.')
print('If you skipped it (small scale, 1 held-out recording), expect this to be inconclusive -- that is itself the point made in Section 6.4.')

mean bits/byte on held-out: 4.959192
real held-out audio:     mean=1.3259 bits/byte, pitch-band z=9.05
shuffled control:        mean=4.9592 bits/byte, pitch-band z=38.38

If you uploaded audio_bundle_full60.zip above, this is the full-scale result the paper needs.
If you skipped it (small scale, 1 held-out recording), expect this to be inconclusive -- that is itself the point made in Section 6.4.


### Fixed-frequency control and harmonic check (audio)

The band-search $z$ above lets the shuffled data choose its own best frequency, which inflates it. The next two cells test the frequency where the *real* data peaks directly on the shuffled data, then check how that peak relates to the context-reset period (255 samples).

In [13]:
sig = audio_losses - audio_losses.mean()
spec = np.abs(np.fft.rfft(sig))
freqs = np.fft.rfftfreq(len(sig), d=1.0)
band = (freqs > 1/200) & (freqs < 1/40)
real_peak_freq = freqs[band][np.argmax(spec[band])]

def fixed_freq_zscore(cost_signal, target_freq):
    sig2 = cost_signal - cost_signal.mean()
    spec2 = np.abs(np.fft.rfft(sig2))
    freqs2 = np.fft.rfftfreq(len(sig2), d=1.0)
    idx = np.argmin(np.abs(freqs2 - target_freq))
    others = np.delete(spec2[1:], idx-1)
    return (spec2[idx]-others.mean())/others.std()

print('real (own peak freq):', fixed_freq_zscore(audio_losses, real_peak_freq))
print('shuffled (same freq, not re-searched):', fixed_freq_zscore(audio_shuf_losses, real_peak_freq))


real (own peak freq): 9.047776
shuffled (same freq, not re-searched): 27.75109


In [14]:
print('real peak period (samples):', 1/real_peak_freq)
print('real peak frequency (Hz):', real_peak_freq * 8000)
print('ratio to context-reset period (255):', 255 / (1/real_peak_freq))

real peak period (samples): 63.75
real peak frequency (Hz): 125.49019607843137
ratio to context-reset period (255): 4.0


### Harmonic sweep

The real peak (63.75 samples) is exactly 255/4. Every harmonic of 255 that falls in the pitch range, tested on real and shuffled data. All of them are stronger under shuffling, which is the signature of an evaluation-window artifact rather than content.

In [15]:
import numpy as np

def fixed_freq_zscore(cost_signal, target_freq):
    sig = cost_signal - cost_signal.mean()
    spec = np.abs(np.fft.rfft(sig))
    freqs = np.fft.rfftfreq(len(sig), d=1.0)
    idx = np.argmin(np.abs(freqs - target_freq))
    others = np.delete(spec[1:], idx-1)
    return (spec[idx]-others.mean())/others.std()

print(f'{"harmonic":>10} {"period":>8} {"z (real)":>10} {"z (shuffled)":>13}')
for k in [2, 3, 4, 5, 6]:
    period = 255 / k
    freq = 1 / period
    if 40 <= period <= 200:
        zr = fixed_freq_zscore(audio_losses, freq)
        zs = fixed_freq_zscore(audio_shuf_losses, freq)
        print(f'{k:>10} {period:>8.2f} {zr:>10.2f} {zs:>13.2f}')


  harmonic   period   z (real)  z (shuffled)
         2   127.50       7.33         38.48
         3    85.00       5.63         35.48
         4    63.75       9.05         27.75
         5    51.00       4.65         27.08
         6    42.50       4.55         24.16


### Reset-corrected test (audio)

Subtracting the mean cost at each within-window position removes the context-reset period and all of its harmonics exactly, leaving only content-dependent variation. If a pitch signal sat on top of the artifact, it would survive this correction.

It does not: at every candidate pitch period (62, 63.75, 64.2, 64.5 samples) $|z| < 2$ for both the real and the shuffled data. The band maximum that remains (182 Hz real, 53 Hz shuffled) matches no plausible pitch and is the usual chance maximum of a wide search band. We therefore report the neural pitch peak as not detected after correction, not as absent.

In [16]:
import numpy as np
def resid(losses, W=255):
    L = losses[:(len(losses)//W)*W].reshape(-1, W)
    return (L - L.mean(axis=0, keepdims=True)).ravel()

def z_at(sig, period):
    s = sig - sig.mean()
    sp = np.abs(np.fft.rfft(s)); f = np.fft.rfftfreq(len(s))
    i = np.argmin(np.abs(f - 1/period))
    others = np.delete(sp[1:], i-1)
    return (sp[i]-others.mean())/others.std()

def band_peak(sig, lo=40, hi=200):
    s = sig - sig.mean()
    sp = np.abs(np.fft.rfft(s)); f = np.fft.rfftfreq(len(s))
    m = (f > 1/hi) & (f < 1/lo)
    return 1/f[m][np.argmax(sp[m])], sp[m].max()

for name, x in [('real', audio_losses), ('shuffled', audio_shuf_losses)]:
    r = resid(x)
    p, _ = band_peak(r)
    print(name, 'peak period after position-mean removal:', p, '-> Hz:', 8000/p)
    for per in [62.0, 63.75, 64.2, 64.5]:
        print('   z at', per, ':', round(z_at(r, per), 2))


real peak period after position-mean removal: 43.87437039199941 -> Hz: 182.33879890522184
   z at 62.0 : 1.04
   z at 63.75 : -0.87
   z at 64.2 : 0.81
   z at 64.5 : 1.74
shuffled peak period after position-mean removal: 151.7381974248927 -> Hz: 52.722387215386796
   z at 62.0 : 1.49
   z at 63.75 : -1.79
   z at 64.2 : -1.05
   z at 64.5 : -0.13


### Reset-corrected test (image)

The same correction for the image model, since the row width (128) sits near the second harmonic of the context-reset period (255/2 = 127.5). The row-width signal survives and strengthens ($z = 18.7$ at 128, none at 127.5) and collapses under shuffling ($z = 0.5$), so it is not a harmonic artifact.

In [17]:
import numpy as np
def resid(losses, W=255):
    L = losses[:(len(losses)//W)*W].reshape(-1, W)
    return (L - L.mean(axis=0, keepdims=True)).ravel()

def z_at(sig, period):
    s = sig - sig.mean()
    sp = np.abs(np.fft.rfft(s)); f = np.fft.rfftfreq(len(s))
    i = np.argmin(np.abs(f - 1/period))
    others = np.delete(sp[1:], i-1)
    return (sp[i]-others.mean())/others.std()

for name, x in [('real', img_losses), ('shuffled', img_shuf_losses)]:
    r = resid(x)
    print(name, 'z at 128:', round(z_at(r, 128), 2), '| z at 127.5:', round(z_at(r, 127.5), 2))


real z at 128: 18.69 | z at 127.5: -0.91
shuffled z at 128: 0.49 | z at 127.5: -1.91
